# 第 1 周末练习 —— 能源系统选型问答（OpenAI + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个偏工程/能源系统的技术问题
- **输出**：清晰、结构化的解释与建议
- **实现要点**：封装 `answer_question(...)`，用 **流式** `stream=True` + `update_display` 边生成边刷新

这是你在课程期间自己也能天天用的工具：把「该建哪种电站」这类问题丢进来对比模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `llm.chat.completions.create(...)` |
| `messages`（system / user） | `week1_ex_system_prompt` + `final_question` |
| 流式输出 `stream=True` | `display` + `update_display` 原地刷新 |
| OpenAI 云端模型 | `MODEL_GPT = 'gpt-4o-mini'` |
| Ollama（OpenAI 兼容） | `base_url=http://localhost:11434/v1` + `llama3.2:1b` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地需 `ollama pull llama3.2:1b` 并启动 Ollama
3. 改写 `final_question` 后，分别跑 GPT 与 Llama 两格，对比回答


## 作者对题意的理解

作者把题目里的 “technical” 理解为「与工程相关」，并做了一个简化工具：在**能源系统**语境下，帮助比较该建设哪种发电厂更合适。


In [10]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display 显示、update_display 原地刷新（流式常用）
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：云端与 Ollama 兼容端点共用同一套 SDK
from openai import OpenAI


In [23]:
# ========== 常量：模型名字集中写在一处 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 小模型：llama3.2:1b 更省内存；需事先 ollama pull 同名模型
MODEL_LLAMA = 'llama3.2:1b'


In [29]:
# ========== system prompt + 统一问答函数（流式刷新显示） ==========

# system prompt：定「能源系统/工程专家」角色，强调只按 CAPEX/OPEX 做成本论证（英文指令不翻译）
week1_ex_system_prompt = """
You are a Energy Systems and Engineering expert that can recommend the construction 
of new power plants in Central Europe based on their CAPEX and OPEX and only based on this.
You show in clear, consise and structured ways, why and when you prefer one type of power plant over another 
sticking to the cost argument as a main factor. 
You are undogmatic and science-based. 
"""

# 统一入口：按传入的 api_key / model / 可选 base_url 创建客户端，并流式展示回答
def answer_question(sel_api_key, selected_model, question, sel_base_url=None):
    # 创建客户端：云端时 sel_base_url=None；本地时传入 Ollama 的 /v1 地址
    llm = OpenAI(base_url=sel_base_url, api_key=sel_api_key)
    # 发起流式 Chat Completions
    stream = llm.chat.completions.create(
        # 选用调用方指定的模型名
        model=selected_model,
        # messages：这里先 user 后 system（顺序保持原样，不「纠正」为常见写法）
        messages=[
            {'role': 'user', 'content':question},
            {'role':'system', 'content':week1_ex_system_prompt}
        ],
        # stream=True：持续返回增量 chunk
        stream=True
    )
    # response：累积目前已收到的全部文本
    response = ""
    # display_id=True：拿到可更新的显示句柄，后续同一 id 刷新而不是新开一块输出
    display_handle = display(Markdown(""), display_id=True)
    # 逐块遍历流式事件
    for chunk in stream:
        # delta.content 可能为 None；用 or '' 避免把 None 拼进字符串
        response += chunk.choices[0].delta.content or ''
        # 用累积后的完整 Markdown 刷新同一显示位
        update_display(Markdown(response), display_id=display_handle.display_id)


In [27]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 真正发给模型的用户问题（英文内容保留；改译会改变模型关注点与回答语言）
final_question = """
Does it make more sense to build solar and wind power plants 
or nuclear power plants in France? Be specific about the conditions in the country and answer in 10 sentences.
"""


In [ ]:
# ========== 调用 gpt-4o-mini：加载密钥后走云端流式问答 ==========

# 加载 .env；override=True 用文件值覆盖进程里已有同名变量
load_dotenv(override=True)
# 把环境变量里的 OPENAI_API_KEY、云端模型名、问题传给统一函数
answer_question(sel_api_key=os.getenv('OPENAI_API_KEY'),selected_model=MODEL_GPT, question=final_question)


In [ ]:
# Jupyter shell magic：拉取本地小模型 llama3.2:1b（需本机已安装并启动 Ollama）
!ollama pull llama3.2:1b


In [ ]:
# ========== 调用本地 Llama：OpenAI 兼容端点 + 同一问题流式问答 ==========

# sel_base_url 指向 Ollama /v1；api_key 占位为 'ollama'；模型用 MODEL_LLAMA
answer_question(sel_base_url="http://localhost:11434/v1", sel_api_key='ollama',selected_model=MODEL_LLAMA, question=final_question)
